## Step 1: Import the libraries

In [2]:
import pandas as pd
import altair as alt

## Step 2: Read the dataset fron the requirements

In [3]:
df = pd.read_csv("licenses_fall2022.csv")
df

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,888281,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129002843.0,NOT RENEWED,N,NaN,JENNIFER,NaN,DARROW,...,NaN,NaN,NaN,N,08/03/2006,NaN,NaN,NaN,NaN,NaN
9996,766623,DETECTIVE BOARD,FIREARM CONTROL CARD,229014180,TERMINATED CARD RETURNED,N,NaN,BRYAN,NaN,WILLIAMS,...,NaN,NaN,NaN,N,08/07/2006,NaN,NaN,NaN,NaN,NaN
9997,399398,COSMO,LICENSED COSMETOLOGIST,11120249,NOT RENEWED,N,NaN,EUGENE,NaN,HENDERSON JR,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
9998,486713,COSMO,LICENSED COSMETOLOGIST,11193270,ACTIVE,N,NaN,MAHLON DOUGLAS,NaN,CLIFT,...,NaN,NaN,NaN,N,12/17/2021,NaN,NaN,NaN,NaN,NaN


## Step 3: Clean the data

In [4]:
df["Original Issue Date"] = pd.to_datetime(df["Original Issue Date"], errors="coerce")

df = df.dropna(subset=["License Type", "Original Issue Date"]).copy()

df["issue_year"] = df["Original Issue Date"].dt.year

## Step 4: Prevent potential Altair row limit

In [5]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Plot 1: Select the top 10 license types

In [6]:
top10 = df["License Type"].value_counts().head(10).reset_index()
top10.columns = ["License Type", "count"]

chart1 = alt.Chart(top10).mark_bar().encode(
    x=alt.X("count:Q", title="Number of Licenses"),
    y=alt.Y("License Type:N", sort="-x", title="License Type"),
    color=alt.Color("License Type:N")
).properties(
    width=600,
    height=400,
    title="Top 10 License Types"
)
chart1

alt.Chart(...)

## Save Plot 1

In [7]:
chart1.save("chart1.json")

## Plot 2: 

In [11]:
top6 = df["License Type"].value_counts().head(6).index.tolist()
year_counts = (
    df[df["License Type"].isin(top6)]
    .groupby(["issue_year", "License Type"])
    .size()
    .reset_index(name="count"))
recent_counts = year_counts[
    (year_counts["issue_year"] >= 2000) & 
    (year_counts["issue_year"] <= 2022)
].copy() # I originally skip this filter years, then I realized that it has too much years to show on the chart, so I did filter to recent years.
license_dropdown = alt.binding_select(
    options=top6,
    name="Choose License Type: "
)
license_select = alt.selection_point(
    fields=["License Type"],
    bind=license_dropdown,
    value=top6[0]
)
chart2 = alt.Chart(recent_counts).mark_line(point=True).encode(
    x=alt.X("issue_year:O", title="Issue Year"),
    y=alt.Y("count:Q", title="Number of Licenses"),
    color="License Type:N",
    tooltip=["License Type", "issue_year", "count"]
).transform_filter(
    license_select
).add_params(
    license_select
).properties(
    width=700,
    height=400,
    title="License Trends Over Time from 2000 to 2022"
)
chart2

alt.Chart(...)

## Save Plot 2

In [12]:
chart2.save("chart2.json")